# 23c · Compressed candidates and exact token rescoring

## The engineering question
How much can a compact candidate representation save, and what cannot exact rescoring repair?

**Learning outcomes:** compare pooled/projection candidates against exhaustive token MaxSim; measure candidate recall separately from relevance; account for the retained full matrices; identify query-space incompatibility.

We encode real ColBERT token matrices, average the rows, and apply a seeded Gaussian projection to 24, 48, or 96 dimensions. This is a deliberately simple lossy baseline. Qdrant searches compact vectors; exact MaxSim reranks 3, 5, or 10 candidates. All widths and budgets are fixed before examining development queries. MaxSim's top-three set is the fidelity oracle, not semantic ground truth.

MUVERA uses **asymmetric fixed-dimensional encodings** whose inner products approximate multivector similarity. A mean of token rows is not that construction and inherits none of its approximation guarantees. Its paper was revised in June 2026 to correct a dimension bound; consult the versioned source rather than repeating an older theorem. [MUVERA v2](https://arxiv.org/abs/2405.19504v2). This lesson reproduces neither its algorithm nor its published benchmark numbers.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("23c", "O", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
from uuid import uuid4
from qdrant_client import QdrantClient, models as qm
from vector_masterclass.late_interaction import encode_tokens, maxsim
from vector_masterclass.data import documents, queries
from vector_masterclass.config import qdrant_url
from vector_masterclass.retrieval import normalized
from vector_masterclass.metrics import ranking_metrics

docs = documents(tenant="public")
qs = queries("development")
ids = [d.key for d in docs]
started = time.perf_counter()
dv = encode_tokens([d.text for d in docs])
qv = encode_tokens([q.text for q in qs], role="query")
encoding_seconds = time.perf_counter() - started
started = time.perf_counter()
oracle = np.array([[maxsim(q, d) for d in dv] for q in qv])
exhaustive_seconds = time.perf_counter() - started
dmean = np.array([v.mean(axis=0) for v in dv])
qmean = np.array([v.mean(axis=0) for v in qv])
rows = []
costs = []
full_bytes = sum(v.nbytes for v in dv)
client = QdrantClient(url=qdrant_url(), timeout=30)
try:
    for width in [24, 48, 96]:
        projection = (
            np.eye(96, dtype=np.float32)
            if width == 96
            else np.random.default_rng(17).normal(size=(96, width)).astype(np.float32)
            / np.sqrt(width)
        )
        compact = normalized(dmean @ projection)
        qcompact = normalized(qmean @ projection)
        costs.append(
            {
                "width": width,
                "compact_bytes": compact.nbytes,
                "full_token_bytes_retained": full_bytes,
                "total_raw_bytes": compact.nbytes + full_bytes,
            }
        )
        name = "vmc_23c_" + uuid4().hex[:8]
        client.create_collection(
            name, vectors_config=qm.VectorParams(size=width, distance=qm.Distance.COSINE)
        )
        try:
            client.upsert(
                name,
                points=[qm.PointStruct(id=i, vector=v.tolist()) for i, v in enumerate(compact)],
                wait=True,
            )
            for qi, (query, qshort) in enumerate(zip(qs, qcompact)):
                truth = sorted(range(len(docs)), key=lambda i: (-oracle[qi, i], ids[i]))[:3]
                for budget in [3, 5, 10]:
                    started = time.perf_counter()
                    hits = client.query_points(
                        name,
                        query=qshort.tolist(),
                        limit=budget,
                        search_params=qm.SearchParams(exact=True),
                    ).points
                    candidate_seconds = time.perf_counter() - started
                    candidates = [h.id for h in hits]
                    started = time.perf_counter()
                    rescored = sorted(candidates, key=lambda i: (-maxsim(qv[qi], dv[i]), ids[i]))[
                        :3
                    ]
                    rescore_seconds = time.perf_counter() - started
                    rows.append(
                        {
                            "width": width,
                            "candidates": budget,
                            "query": query.id,
                            "candidate_recall_of_MaxSim_top3": len(set(truth) & set(candidates))
                            / 3,
                            "candidate_seconds": candidate_seconds,
                            "rescore_seconds": rescore_seconds,
                            "token_dot_products": sum(len(qv[qi]) * len(dv[i]) for i in candidates),
                            "exhaustive_token_dot_products": sum(len(qv[qi]) * len(d) for d in dv),
                            **ranking_metrics([ids[i] for i in rescored], query.relevance, 3),
                        }
                    )
        finally:
            client.delete_collection(name)
finally:
    client.close()
results = lesson.table(pd.DataFrame(rows))
costs = lesson.table(pd.DataFrame(costs), "costs")
summary = results.groupby(["width", "candidates"])[
    [
        "candidate_recall_of_MaxSim_top3",
        "ndcg",
        "token_dot_products",
        "candidate_seconds",
        "rescore_seconds",
    ]
].mean()
display(summary)
display(costs)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for width, group in results.groupby("width"):
    aggregate = group.groupby("candidates").mean(numeric_only=True)
    axes[0].plot(
        aggregate.index, aggregate.candidate_recall_of_MaxSim_top3, "o-", label=str(width) + "D"
    )
    axes[1].plot(aggregate.index, aggregate.ndcg, "o-", label=str(width) + "D")
axes[0].set(
    xlabel="Candidate budget",
    ylabel="Mean oracle candidate recall",
    ylim=(0, 1.05),
    title="Rescoring cannot recover absent candidates",
)
axes[1].set(
    xlabel="Candidate budget",
    ylabel="Development nDCG@3",
    ylim=(0, 1.05),
    title="Fidelity and semantic quality differ",
)
for ax in axes:
    ax.legend()
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Compact vectors add an index; they do not eliminate the full matrices needed for rescoring. The tiny exact Qdrant scan includes HTTP overhead and cannot estimate million-document ANN throughput. Token dot-product counts expose algorithmic work without pretending that counts equal wall time. Candidate recall can increase while labeled relevance does not: an exact model oracle may itself rank the wrong answer.

**Exercise:** hold raw compact-vector bytes fixed and compare a second projection seed on development data. Select a budget using a quality constraint, then evaluate a newly reserved holdout. Explain why symmetric pooling loses token-level query coverage, and which MUVERA claims would require implementing and validating its actual asymmetric encoder.

In [ ]:
manifest = lesson.finish(
    {
        "nine_configurations": len(results) == len(qs) * 9,
        "candidate_recall_bounded": bool(
            results.candidate_recall_of_MaxSim_top3.between(0, 1).all()
        ),
        "bounded_rescoring_work": bool(
            (results.token_dot_products <= results.exhaustive_token_dot_products).all()
        ),
        "full_matrices_accounted": bool((costs.total_raw_bytes > full_bytes).all()),
    },
    observations={
        "encoding_seconds": encoding_seconds,
        "exhaustive_MaxSim_seconds": exhaustive_seconds,
        "split": "development",
        "algorithm": "mean token pooling plus seeded random projection, exact candidate retrieval, MaxSim rescore",
        "MUVERA_reproduction": False,
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.